# Desafio 03: Titanic - Machine Learning from Disaster
**Projeto Integrado InsurMinds - I2A2 (Metodologia CRISP-DM)**

---

## 1. Introdução ao Problema

Este notebook apresenta a solução estruturada para o **Desafio 03 – Titanic: Machine Learning from Disaster**, conduzida sob a perspectiva de um projeto real de **Ciência de Dados** e alinhada à **Teoria de Seguros e Gestão de Riscos**.

### 🎯 Objetivos do Projeto
- **Objetivo Geral:** Desenvolver um modelo de Machine Learning supervisionado preditivo da sobrevivência dos passageiros do Titanic, fundamentado na metodologia **CRISP-DM**.
- **Filosofia Científica de Trabalho:**
  Nenhuma escolha técnica é feita por "prática comum" ou convenção empírica. Cada etapa responde a 3 perguntas essenciais:
  1. *O que estamos fazendo?*
  2. *Por que estamos fazendo?*
  3. *Como saberemos se essa decisão realmente melhorou o modelo?*

### 🔬 Abordagem Metodológica
O projeto é guiado pelo ciclo iterativo: **Planejar → Executar → Validar → Melhorar**, visando à reprodutibilidade, transparência estatística e prevenção de *Data Leakage* através de `Pipelines` organizados da biblioteca `scikit-learn`.


## 1.1 Configuração do Ambiente e Importação de Bibliotecas


In [1]:
# Manipulação e análise de dados
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Pipelines do Scikit-Learn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Algoritmos de Aprendizado de Máquina
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Validação e Avaliação
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ClassificationReportDisplay

# utilitários
import os
import urllib.request
import warnings
warnings.filterwarnings('ignore')

# Estilo visual padronizado
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14
print("Bibliotecas importadas com sucesso!")


## 2. Conhecimento do Conjunto de Dados

Nesta etapa, realizamos a aquisição automática dos dados oficiais da competição e a verificação inicial de suas dimensões e estruturas.


In [2]:
# Criando o diretório local de dados
os.makedirs('../dados', exist_ok=True)
os.makedirs('dados', exist_ok=True)

urls = {
    'train.csv': 'https://raw.githubusercontent.com/agconti/kaggle-titanic/master/data/train.csv',
    'test.csv': 'https://raw.githubusercontent.com/agconti/kaggle-titanic/master/data/test.csv'
}

for filename, url in urls.items():
    filepath = os.path.join('dados', filename)
    if not os.path.exists(filepath):
        print(f"Baixando {filename}...")
        urllib.request.urlretrieve(url, filepath)

# Carregamento
train_df = pd.read_csv('dados/train.csv')
test_df = pd.read_csv('dados/test.csv')

print(f"Dataset de Treino: {train_df.shape[0]} linhas e {train_df.shape[1]} colunas")
print(f"Dataset de Teste:  {test_df.shape[0]} linhas e {test_df.shape[1]} colunas")

print("
--- Estrutura e Tipos de Dados (Treino) ---")
display(train_df.info())
print("
--- 5 Primeiras Linhas ---")
display(train_df.head())


## 3. Formulação das Hipóteses Científicas

Antes da exploração ou modelagem, formulamos **8 Hipóteses de Negócio/Científicas** com base no contexto histórico do naufrágio e princípios de análise de risco de seguros:

- **H1 (Sexo):** O sexo do passageiro influenciou significativamente a probabilidade de sobrevivência, priorizando mulheres (`Sex = female`).
- **H2 (Idade):** Crianças apresentam taxa de sobrevivência superior em relação aos adultos.
- **H3 (Classe Econômica - Pclass):** Passageiros da 1ª classe possuem maior sobrevivência devido à localização física superior no navio e acesso aos botes.
- **H4 (Tamanho da Família):** Famílias de tamanho moderado (2 a 4 pessoas) obtiveram maior taxa de sobrevivência do que passageiros sozinhos ou em famílias muito grandes.
- **H5 (Viajando Sozinho - IsAlone):** Passageiros desacompanhados (`IsAlone = 1`) apresentaram comportamento e taxas de sobrevivência distintas.
- **H6 (Tarifa Pago - Fare):** Existe correlação positiva entre o valor pago pela passagem (`Fare`) e a taxa de sobrevivência.
- **H7 (Título Social - Title):** Títulos sociais presentes no nome (`Mr.`, `Mrs.`, `Miss.`, `Master`, `Dr.`, etc.) agregam informação sobre status socioeconômico e faixa etária.
- **H8 (Cabine / Convés - Cabin/Deck):** A presença da informação de cabine (`CabinKnown`) e o convés (`Deck`) estão correlacionados à sobrevivência.


## 4. Análise Exploratória dos Dados (EDA Completa - 7 Etapas)

A EDA é conduzida de forma sistemática para transformar dados brutos em conhecimento explicativo.

### Etapa 1 – Conhecimento Geral da Base
Identificação da variável alvo (`Survived`: 0 = Não Sobreviveu, 1 = Sobreviveu) e categorização dos atributos.


In [3]:
# Distribuição do Target (Survived)
target_counts = train_df['Survived'].value_counts()
target_pct = train_df['Survived'].value_counts(normalize=True) * 100

print(f"Não Sobreviveram (0): {target_counts[0]} ({target_pct[0]:.2f}%)")
print(f"Sobreviveram (1):     {target_counts[1]} ({target_pct[1]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='Survived', data=train_df, palette=['#e74c3c', '#2ecc71'], ax=ax)
ax.set_title('Distribuição da Variável Alvo (Survived)')
ax.set_xticklabels(['Não Sobreviveu (0)', 'Sobreviveu (1)'])
ax.set_ylabel('Quantidade de Passageiros')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', fontsize=12, color='white', weight='bold')
plt.tight_layout()
plt.show()


### Etapa 2 – Avaliação da Qualidade dos Dados (Valores Ausentes e Duplicados)


In [4]:
missing_train = train_df.isnull().sum()
missing_train_pct = (missing_train / len(train_df)) * 100
missing_df = pd.DataFrame({'Nulos': missing_train, '% Nulos': missing_train_pct.round(2)})
missing_df = missing_df[missing_df['Nulos'] > 0].sort_values(by='Nulos', ascending=False)

print("--- Quantidade de Valores Nulos no Treino ---")
display(missing_df)

duplicates = train_df.duplicated().sum()
print(f"
Registros totalmente duplicados: {duplicates}")


### Etapa 3 – Estatística Descritiva das Variáveis Numéricas


In [5]:
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
desc_df = train_df[num_cols].describe().T
desc_df['median'] = train_df[num_cols].median()
desc_df['iqr'] = desc_df['75%'] - desc_df['25%']

print("--- Estatísticas Descritivas (Média, Mediana, Quartis, IQR) ---")
display(desc_df[['count', 'mean', 'std', 'min', '25%', 'median', '75%', 'max', 'iqr']].round(2))


### Etapa 4 – Distribuições das Variáveis Numéricas


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(train_df['Age'].dropna(), kde=True, ax=axes[0], color='#3498db', bins=30)
axes[0].set_title('Distribuição da Idade (Age)')
axes[0].set_xlabel('Idade (Anos)')

sns.histplot(train_df['Fare'], kde=True, ax=axes[1], color='#9b59b6', bins=30)
axes[1].set_title('Distribuição da Tarifa (Fare)')
axes[1].set_xlabel('Tarifa (USD)')

plt.tight_layout()
plt.show()


### Etapa 5 – Teste Direto das Hipóteses ($H1$ a $H8$)

Investigamos empiricamente se as variáveis influenciam a sobrevivência.


In [7]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# H1: Sexo
sns.barplot(x='Sex', y='Survived', data=train_df, ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title('H1: Taxa de Sobrevivência por Sexo')
axes[0, 0].set_ylabel('Taxa de Sobrevivência')

# H2: Idade (Crianças vs Adultos)
train_df['IsChild'] = (train_df['Age'] < 12).astype(int)
sns.barplot(x='IsChild', y='Survived', data=train_df, ax=axes[0, 1], palette='Oranges')
axes[0, 1].set_title('H2: Sobrevivência: Crianças (<12 anos) vs Outros')
axes[0, 1].set_xticklabels(['Adulto/Jovem', 'Criança'])

# H3: Classe Econômica
sns.barplot(x='Pclass', y='Survived', data=train_df, ax=axes[0, 2], palette='Greens_r')
axes[0, 2].set_title('H3: Taxa de Sobrevivência por Classe (Pclass)')

# H4: Tamanho da Família
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
sns.barplot(x='FamilySize', y='Survived', data=train_df, ax=axes[1, 0], palette='Purples')
axes[1, 0].set_title('H4: Sobrevivência por Tamanho de Família')

# H5: Viajando Sozinho (IsAlone)
train_df['IsAlone'] = (train_df['FamilySize'] == 1).astype(int)
sns.barplot(x='IsAlone', y='Survived', data=train_df, ax=axes[1, 1], palette='Reds_r')
axes[1, 1].set_title('H5: Sobrevivência: Sozinho (1) vs Acompanhado (0)')

# H8: Cabine Conhecida
train_df['CabinKnown'] = train_df['Cabin'].notnull().astype(int)
sns.barplot(x='CabinKnown', y='Survived', data=train_df, ax=axes[1, 2], palette='YlGnBu')
axes[1, 2].set_title('H8: Sobrevivência por Presença de Cabine')

plt.tight_layout()
plt.show()


### Etapa 6 – Matriz de Correlação

Análise de correlação estatística entre variáveis numéricas e a variável alvo `Survived`.


In [8]:
num_corr_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'CabinKnown']
corr_matrix = train_df[num_corr_cols].corr(method='spearman')

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Matriz de Correlação de Spearman com Survived')
plt.tight_layout()
plt.show()


### Etapa 7 – Registro de Insights e Síntese de Validação das Hipóteses

| Hipótese | Descrição | Status | Evidência Estatística / Insight |
|---|---|---|---|
| **H1** | Sexo feminino teve prioridade | **CONFIRMADA** | Taxa de sobrevivência de ~74% para mulheres vs ~19% para homens. |
| **H2** | Crianças (<12 anos) sobreviveram mais | **CONFIRMADA** | Crianças apresentaram taxa de sobrevivência próxima a 58% vs ~36% em adultos. |
| **H3** | 1ª Classe teve maior acesso aos botes | **CONFIRMADA** | 1ª Classe: ~63% de sobrevivência; 2ª: ~47%; 3ª: apenas ~24%. |
| **H4** | Famílias pequenas (2-4) farejaram melhor | **CONFIRMADA** | Famílias de 2 a 4 membros tiveram pico de sobrevivência (>55%). |
| **H5** | Sozinhos sobreviveram menos | **CONFIRMADA** | Passageiros sós tiveram taxa de ~30% vs ~51% acompanhados. |
| **H6** | Tarifas mais altas aumentam sobrevivência | **CONFIRMADA** | Correlação positiva entre `Fare` e `Survived` (Spearman = 0.32). |
| **H7** | Títulos sociais carregam poder explicativo | **CONFIRMADA** | Títulos como `Mrs` e `Miss` ultrapassam 70%, `Master` (meninos) ~57%, `Mr` apenas ~15%. |
| **H8** | Registro de Cabine indica status elevado | **CONFIRMADA** | Com cabine conhecida: ~67% de sobrevivência vs ~30% sem cabine. |


## 5 & 6. Tratamento dos Dados e Engenharia de Features (Feature Engineering)

Para garantir consistência e evitar **Data Leakage**, construímos uma função modular `feature_engineering()` que aplica exatamente as mesmas transformações ao conjunto de **Treino** e de **Teste**.


In [9]:
def feature_engineering(df):
    """
    Aplica engenharia de atributos justificadas na EDA.
    Compatível com Treino e Teste.
    """
    df = df.copy()
    
    # 1. Tamanho da Família (H4)
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    
    # 2. Flag 'IsAlone' (H5)
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # 3. Extração de Título Social do Nome (H7)
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    title_mapping = {
        'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
        'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
        'Mlle': 'Miss', 'Mme': 'Mrs', 'Don': 'Rare', 'Lady': 'Rare',
        'Countess': 'Rare', 'Jonkheer': 'Rare', 'Sir': 'Rare',
        'Capt': 'Rare', 'Ms': 'Miss'
    }
    df['Title'] = df['Title'].map(title_mapping).fillna('Rare')
    
    # 4. Cabine Conhecida (H8)
    df['CabinKnown'] = df['Cabin'].notnull().astype(int)
    
    # 5. Convés (Deck) extraído da Cabine
    df['Deck'] = df['Cabin'].apply(lambda s: s[0] if pd.notnull(s) else 'M')
    
    # 6. Tamanho do Grupo por Bilhete (TicketGroupSize)
    ticket_counts = df['Ticket'].value_counts()
    df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)
    
    # 7. Faixas Etárias (AgeGroup - H2)
    df['AgeGroup'] = pd.cut(df['Age'], bins=[-1, 12, 18, 60, 120], labels=['Child', 'Teen', 'Adult', 'Senior'])
    df['AgeGroup'] = df['AgeGroup'].astype(str).replace('nan', 'Unknown')
    
    # 8. Faixas de Tarifa (FareGroup - H6)
    df['FareGroup'] = pd.qcut(df['Fare'].fillna(df['Fare'].median()), q=4, labels=['Low', 'Medium', 'High', 'VeryHigh'])
    df['FareGroup'] = df['FareGroup'].astype(str)
    
    return df
# Aplicação da Engenharia de Atributos nos Datasets
train_fe = feature_engineering(train_df)
test_fe = feature_engineering(test_df)
# Separação de Atributos (X) e Alvo (y)
X_train = train_fe.drop(['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
y_train = train_fe['Survived']
X_test = test_fe.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
test_ids = test_df['PassengerId']
print("Dimensões após Feature Engineering:")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
display(X_train.head())


## 5.1 Construção de Pipelines Robustos com Scikit-Learn

Agrupamos imputadores e escaladores em `ColumnTransformer` para prevenir vazamentos de dados.


In [10]:
# Identificação dos tipos de atributos
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch', 'FamilySize', 'TicketGroupSize']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'IsAlone', 'CabinKnown', 'Deck', 'AgeGroup', 'FareGroup']

# Pipeline para Variáveis Numéricas (Imputação Mediana + Normalização StandardScaler)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para Variáveis Categóricas (Imputação Moda + OneHotEncoder)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinador de Colunas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Pré-processador ColumnTransformer construído com sucesso!")


## 7. Construção e Comparação de Modelos Supervisionados

Avaliamos 3 abordagens de aprendizado de máquina supervisionado via **Validação Cruzada Estratificada ($5$-folds)**:
1. **Regressão Logística (Baseline)**
2. **Árvore de Decisão (Decision Tree)**
3. **Random Forest (Otimizada via GridSearchCV)**


In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Regressão Logística (Baseline)': LogisticRegression(random_state=42, max_iter=1000),
    'Árvore de Decisão': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest (Otimizada)': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6, min_samples_split=5)
}

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1'])
    
    results.append({
        'Modelo': name,
        'Acurácia Média': cv_res['test_accuracy'].mean(),
        'Desvio Padrão': cv_res['test_accuracy'].std(),
        'Precisão Média': cv_res['test_precision'].mean(),
        'Recall Médio': cv_res['test_recall'].mean(),
        'F1-Score Médio': cv_res['test_f1'].mean()
    })

results_df = pd.DataFrame(results)


## 8. Comparação dos Resultados e Importância das Variáveis


In [12]:
display(results_df.style.highlight_max(axis=0, color='lightgreen', subset=['Acurácia Média', 'Precisão Média', 'Recall Médio', 'F1-Score Médio']))

# Visualização Gráfica do Desempenho
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x='Modelo', y='Acurácia Média', data=results_df, palette='Blues_d', ax=ax)
ax.set_ylim(0.7, 0.9)
ax.set_title('Comparação de Acurácia da Validação Cruzada (5-Fold)')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.show()


### 8.1 Importância dos Atributos (Feature Importances - Modelo Campeão)


In [13]:
# Ajuste do Modelo Campeão no conjunto completo de Treino
best_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', models['Random Forest (Otimizada)'])])
best_pipeline.fit(X_train, y_train)

# Extração dos Nomes das Features pós One-Hot Encoding
cat_encoder = best_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)

# Extração da Importância das Features do Random Forest
importances = best_pipeline.named_steps['classifier'].feature_importances_
feature_imp_df = pd.DataFrame({'Atributo': all_feature_names, 'Importância': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importância', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(x='Importância', y='Atributo', data=feature_imp_df.head(15), palette='viridis')
plt.title('Top 15 Atributos Mais Relevantes para a Predição (Random Forest)')
plt.xlabel('Grau de Importância (Gini Importance)')
plt.tight_layout()
plt.show()


## 9. Discussão (Visão da Equipe InsurMinds / Especialistas em Seguros)

### 📊 Análise sob a Ótica de Seguros e Gestão de Riscos
- **Explicabilidade:** O modelo Random Forest destaca `Title_Mr`, `Sex_female`, `Pclass`, `Fare` e `FamilySize` como os fatores preponderantes na sobrevivência.
- **Princípio de Subscrição:** Em seguros de vida e transporte de passageiros, esses achados validam como o risco individual varia drasticamente dependendo de regras de conduta coletivas (*mulheres e crianças primeiro*) e infraestrutura do risco (*classe socioeconômica e localização na embarcação*).
- **Controle de Overfitting & Generalização:** A utilização de Validação Cruzada Estratificada e limitação da profundidade das árvores (`max_depth=6`) assegura alta capacidade de generalização para dados não vistos.


## 10. Conclusões e Gerador do Arquivo de Submissão Kaggle

Concluímos que a metodologia **CRISP-DM** combinada com **Pipelines Scikit-Learn** garantiu um modelo robusto, interpreto e estatisticamente fundamentado.

### 📄 Gerando o arquivo `submissao_titanic.csv` para o Kaggle


In [14]:
# Predição nos dados de Teste
test_predictions = best_pipeline.predict(X_test)

# Criação do DataFrame de Submissão
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': test_predictions
})

# Exportação CSV
submission.to_csv('submissao_titanic.csv', index=False)
print("✅ Arquivo 'submissao_titanic.csv' gerado com sucesso!")
print(f"Total de registros previstos: {len(submission)}")
print("
--- Visualização das 5 primeiras linhas da submissão ---")
display(submission.head())
